# Computer Vision — Session 1 Hands-on

## Exploring How Computers See Images

> In this notebook, we will explore how computers represent and work with images. We will load an image, inspect its dimensions and channels, visualize its pixels, resize and crop it, convert it to grayscale, and inspect individual pixel values.

This notebook is a **lab**, not a lecture. Run every cell, read the output, and change the numbers to see what happens.

---

### Learning Objectives

By the end of this notebook, you should be able to:

* Load an image using OpenCV
* Display an image using Matplotlib
* Understand image dimensions
* Inspect image shape
* Understand image channels
* Compare RGB and grayscale images
* Resize an image
* Crop a region from an image
* Inspect individual pixels
* Understand that an image is represented as a NumPy array

---

### The Idea We Are Chasing

```text
Real World
    v
Camera
    v
Image
    v
Pixels
    v
Numbers
    v
NumPy Array
    v
Computer Vision
```

Everything below is about one question: **what does an image actually look like to a computer?**

---

# 2. Setup

We only need three libraries today:

* **OpenCV** (`cv2`) - reads images from files and manipulates them
* **NumPy** (`np`) - lets us treat images as arrays of numbers
* **Matplotlib** (`plt`) - draws images so we can look at them

If a library is missing, install it once from a terminal:
`pip install opencv-python numpy matplotlib`

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Setup complete.")

No error message above means all three libraries are ready to use.

---

# 3. Getting a Sample Image

We need an image file to work with.

The cell below checks whether `sample_image.jpg` exists in this folder. If it does not, it **draws one for us** using NumPy and OpenCV, so the notebook always works.

This is also a first sneak preview of the big idea: we are building a picture by **writing numbers into an array**.

In [ ]:
import os  # part of Python, used here only to check whether a file exists

IMAGE_PATH = "sample_image.jpg"

if not os.path.exists(IMAGE_PATH):
    # An empty picture: 480 rows (height) x 640 columns (width) x 3 channels
    # dtype=np.uint8 means every number must stay between 0 and 255
    canvas = np.zeros((480, 640, 3), dtype=np.uint8)

    # Careful: OpenCV writes colors as (Blue, Green, Red)
    canvas[:280, :] = (230, 180, 90)   # top part    -> light blue sky
    canvas[280:, :] = (60, 140, 60)    # bottom part -> green grass

    cv2.circle(canvas, (520, 90), 55, (60, 220, 250), -1)              # yellow sun
    cv2.rectangle(canvas, (80, 300), (240, 440), (40, 40, 200), -1)    # red house
    cv2.rectangle(canvas, (300, 340), (420, 440), (180, 60, 40), -1)   # blue box
    cv2.rectangle(canvas, (450, 260), (600, 440), (30, 30, 30), -1)    # dark box

    cv2.imwrite(IMAGE_PATH, canvas)   # save the array to disk as a real image file
    print("Sample image created:", IMAGE_PATH)
else:
    print("Using the image already in this folder:", IMAGE_PATH)

**Want to use your own photo?** Put a `.jpg` or `.png` file in the same folder as this notebook and change
`IMAGE_PATH` to its file name. Everything below will work the same way.

---

# 4. Loading an Image

`cv2.imread()` opens the file and hands us back **numbers**, not a picture.

In [ ]:
image = cv2.imread(IMAGE_PATH)

# Always check! OpenCV does not raise an error for a missing or unreadable file.
if image is None:
    print("Error: Image could not be loaded.")
else:
    print("Image loaded successfully.")

### Why check for `None`?

If you misspell the file name, or the file is in another folder, OpenCV does **not** crash.
It quietly returns `None`, and every cell after that fails with a confusing error.
Checking once here saves a lot of debugging later.

In [ ]:
print(type(image))

The output is:

```text
<class 'numpy.ndarray'>
```

> The computer does not store the image as "a picture." It stores it as an array of numbers.

`ndarray` is NumPy's word for "n-dimensional array" - a grid of numbers.
An image is simply a grid where every number describes how bright one color is at one tiny location.

---

# 5. Visualizing the Image

OpenCV has its own display function:

```python
cv2.imshow("My Image", image)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

It opens a **separate desktop window**, which often freezes inside Jupyter notebooks.
So in this course we display images with **Matplotlib** instead.

But there is one detail that confuses almost every beginner:

> OpenCV loads color images using **BGR** channel order, while Matplotlib expects **RGB**.

Same numbers, different order. If we forget to convert, red and blue swap places.

In [ ]:
# cvtColor = "convert color". It reorders the channels from BGR to RGB.
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image)          # BGR data shown as if it were RGB -> wrong colors
axes[0].set_title("Wrong: BGR array shown directly")
axes[0].axis("off")

axes[1].imshow(image_rgb)      # converted first -> correct colors
axes[1].set_title("Correct: converted to RGB")
axes[1].axis("off")

plt.show()

Look carefully: the sky and the red shape changed color between the two pictures.
Nothing about the image changed except **the order of the three numbers in each pixel**.

Now let us display the correct version on its own.

In [ ]:
plt.figure(figsize=(8, 6))   # create a drawing area, 8 x 6 inches
plt.imshow(image_rgb)        # draw the array as a picture
plt.title("Original Image")  # a meaningful title, always
plt.axis("off")              # hide the number rulers on the edges
plt.show()                   # render it

**Tip:** try removing `plt.axis("off")` and running the cell again.
Matplotlib will show the pixel coordinates along the edges - useful when hunting for a specific pixel.

---

# 6. Understanding Image Dimensions

Every NumPy array knows its own size. We read it with `.shape`.

In [ ]:
print(image.shape)

You should see something like:

```text
(480, 640, 3)
```

Read it as:

```text
480 -> Height   (number of rows of pixels)
640 -> Width    (number of columns of pixels)
3   -> Channels (Blue, Green, Red)
```

**Height comes first.** This trips up beginners constantly, because we usually say
"640 by 480" when talking about screens. NumPy thinks in rows first, then columns.

In [ ]:
height, width, channels = image.shape

print(f"Height: {height} pixels")
print(f"Width: {width} pixels")
print(f"Channels: {channels}")

> Height and width tell us how many pixels the image contains in each direction.

So how many pixels are there in total?

In [ ]:
total_pixels = height * width

print(f"Total pixels: {total_pixels}")
print(f"Total numbers stored: {total_pixels * channels}")

Every one of those numbers is between 0 and 255.
A "small" image already holds hundreds of thousands of numbers - which is exactly why we let computers do the looking.

### Quick question

If an image had shape `(1080, 1920, 3)`, what are its height, width, and channel count?
How many pixels does it contain?

---

# 7. Understanding Image Channels

A color image is really **three grayscale images stacked together**:

* one holding how much **Blue** each pixel has
* one holding how much **Green** each pixel has
* one holding how much **Red** each pixel has

Because OpenCV stores them in BGR order, the index of each channel is:

```text
image[:, :, 0] -> Blue
image[:, :, 1] -> Green
image[:, :, 2] -> Red
```

The `:` means "all of them", so `image[:, :, 0]` reads as
"all rows, all columns, channel number 0".

In [ ]:
blue_channel  = image[:, :, 0]
green_channel = image[:, :, 1]
red_channel   = image[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Image")

axes[1].imshow(red_channel, cmap="gray")
axes[1].set_title("Red Channel")

axes[2].imshow(green_channel, cmap="gray")
axes[2].set_title("Green Channel")

axes[3].imshow(blue_channel, cmap="gray")
axes[3].set_title("Blue Channel")

for ax in axes:
    ax.axis("off")

plt.show()

### What are we looking at?

Each channel is drawn in gray on purpose, because a single channel holds **only one number per pixel** -
there is no color information left in it, just an amount.

* **White / bright** = a high number (close to 255) = a lot of that color here
* **Black / dark** = a low number (close to 0) = almost none of that color here

Find the red shape in the original image, then look at the same spot in the three channels:
it glows in the Red channel and goes dark in the other two. That *is* what "red" means to a computer.

In [ ]:
print("Full color image shape:", image.shape)
print("Blue channel shape:    ", blue_channel.shape)

```text
Color image   -> Height x Width x 3
Single channel -> Height x Width
```

Picking one channel removes the third dimension. We went from a stack of three grids down to a single grid.

---

# 8. Inspecting the Image as a NumPy Array

Time to look at the actual numbers.

We will print only the **top-left 5 x 5 corner** of the image.
Never print a whole image - that is hundreds of thousands of numbers scrolling past.

In [ ]:
print(image[:5, :5])

### How to read that output

* The outer brackets group the **rows** (5 of them).
* Inside each row there are **5 pixels**.
* Inside each pixel there are **3 numbers**: `[B, G, R]`.

So the whole thing is a small block of the picture, written out as pure numbers.

In our sample image the top-left corner is a flat patch of sky, so all 25 pixels hold **the same** three numbers.
Try `print(image[298:303, 78:83])` instead - that block sits right on the corner where the grass meets the red shape, and the numbers change from pixel to pixel.

Let us zoom in on a single pixel.

In [ ]:
pixel = image[100, 100]

print("Pixel from the BGR image:", pixel)

With OpenCV, the values are ordered as:

```text
[B, G, R]
```

Now the same pixel, read from the RGB version of the image:

In [ ]:
pixel_rgb = image_rgb[100, 100]

print("Pixel from the BGR image:", pixel)
print("Pixel from the RGB image:", pixel_rgb)

### BGR vs RGB

```text
BGR -> [Blue, Green, Red]   (what OpenCV gives you)
RGB -> [Red, Green, Blue]   (what Matplotlib and most of the world expect)
```

The two printed lines contain **exactly the same three numbers, in reverse order**.
The pixel did not change - only the way we chose to write it down.

This is the single most common source of "why are my colors weird?" bugs in OpenCV.

---

## Let's Inspect a Pixel

Pick a location and find out exactly what the computer stores there.

We use two names:

* `x` - how far **across** (column)
* `y` - how far **down** (row)

But NumPy indexing is written:

```python
image[y, x]
```

**`y` first, then `x`** - because the first dimension of the array is rows (vertical),
and the second is columns (horizontal). It feels backwards at first. It stops feeling backwards with practice.

In [ ]:
x = 200   # 200 pixels from the left edge
y = 150   # 150 pixels from the top edge

pixel = image[y, x]   # remember: [row, column] = [y, x]

blue, green, red = pixel

print(f"Pixel at (x={x}, y={y})")
print(f"B value: {blue}")
print(f"G value: {green}")
print(f"R value: {red}")

Those three numbers describe **one tiny square** of the picture - one of the hundreds of thousands we counted earlier.
The whole image is nothing more than this, repeated over and over in a grid.

Let us mark that pixel so we can see where it lives.

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)

# scatter() draws a marker at a point, given as (x, y) - Matplotlib's order,
# which is the opposite of NumPy's [y, x] indexing.
plt.scatter([x], [y], s=250, facecolors="none", edgecolors="red", linewidths=2)

plt.title(f"Selected Pixel at (x={x}, y={y})")
plt.show()

Notice the axis numbers along the edges: across the top is `x`, down the side is `y`,
and `(0, 0)` sits in the **top-left corner** - not the bottom-left like in maths class.

### Challenge

> Choose a different `(x, y)` coordinate and inspect the pixel value. Try selecting a point from a dark area
> and then from a bright or colorful area. What changes?

Good spots to try in the sample image: `(520, 90)` on the sun, `(500, 350)` on the dark box, `(150, 380)` on the red shape.

---

# 9. Resizing an Image

> Sometimes we need an image to have a different width and height.

Photos come in every size imaginable, and later on we will want them all the same size.
`cv2.resize()` does that.

One trap:

```text
cv2.resize() expects (width, height)
image.shape  gives   (height, width, channels)
```

The order is flipped. Read the size you pass in carefully.

In [ ]:
resized_image = cv2.resize(image_rgb, (300, 200))   # (width, height)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image_rgb)
axes[0].set_title(f"Original - {image_rgb.shape[1]} x {image_rgb.shape[0]}")
axes[0].axis("off")

axes[1].imshow(resized_image)
axes[1].set_title(f"Resized - {resized_image.shape[1]} x {resized_image.shape[0]}")
axes[1].axis("off")

plt.show()

In [ ]:
print("Original:", image_rgb.shape)
print("Resized:", resized_image.shape)

Both pictures look similar on screen because Matplotlib stretches them to fill their boxes.
The **shapes** tell the real story: the resized image contains far fewer numbers.

Resizing is not free - shrinking an image throws information away permanently.

### Challenge

> Try resizing the image to a different size. What happens to the shape?

Try `(64, 64)` and display it. At what point does the picture stop being recognisable?

---

# 10. Cropping an Image

> Cropping means selecting only a specific region of the image.

There is no `cv2.crop()` function - and we do not need one.
An image is a NumPy array, so we just **slice** it, exactly like slicing a list.

```python
image[start_y:end_y, start_x:end_x]
```

Once again, **`y` comes before `x`**, because the first dimension of the array is rows
(how far down) and the second is columns (how far across). Cropping is really "keep these rows, keep these columns".

In [ ]:
cropped_image = image_rgb[100:400, 150:500]   # rows 100-399, columns 150-499

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(cropped_image)
axes[1].set_title("Cropped Region")
axes[1].axis("off")

plt.show()

print("Original:", image_rgb.shape)
print("Cropped:", cropped_image.shape)

Check the numbers: `400 - 100 = 300` rows and `500 - 150 = 350` columns, so the cropped shape is `(300, 350, 3)`.
The height and width changed; the channel count did not, because a cropped color image is still a color image.

### Challenge

> Try cropping a different region of the image.

Can you crop out just the sun? (Hint: it is centred near `x = 520`, `y = 90`.)

---

# 11. RGB to Grayscale

> An RGB image has three channels. A grayscale image represents intensity using one channel.

In a grayscale image each pixel holds a **single number**: 0 is black, 255 is white,
and everything in between is a shade of gray. Color is gone; brightness remains.

In [ ]:
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

print("RGB Image Shape:", image_rgb.shape)
print("Grayscale Image Shape:", gray_image.shape)

```text
RGB       -> Height x Width x 3
Grayscale -> Height x Width
```

The third dimension disappeared. A grayscale image stores **one third** of the numbers a color image does.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image_rgb)
axes[0].set_title("Original RGB Image")
axes[0].axis("off")

# cmap="gray" tells Matplotlib to draw these single numbers as shades of gray.
# Without it, Matplotlib would invent a colorful palette of its own.
axes[1].imshow(gray_image, cmap="gray")
axes[1].set_title("Grayscale Image")
axes[1].axis("off")

plt.show()

In [ ]:
print("One RGB pixel:      ", image_rgb[150, 200], "-> three numbers")
print("The same gray pixel:", gray_image[150, 200], "-> one number")

Two very different colors can end up with the same gray value, because they are equally bright.
That is the trade-off: grayscale is smaller and simpler, but some information is lost forever.

---

# Mini Project - Build Your Image Explorer

Time to put everything together.

Below is one function that performs the whole tour: load, inspect, convert, resize, crop, and report.
Nothing in it is new - every line comes from a section you already ran.

Read it once from top to bottom, then run it and compare the printed report with the pictures.

In [ ]:
def explore_image(image_path, pixel_x=200, pixel_y=150,
                  new_size=(300, 200), crop_box=(100, 400, 150, 500)):
    # crop_box is (start_y, end_y, start_x, end_x)

    # 1. Load the image
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Image could not be loaded ->", image_path)
        return

    # 2. Make the versions we need
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3. Basic information
    height, width, channels = img.shape

    # 4. Resize and crop
    img_resized = cv2.resize(img_rgb, new_size)
    start_y, end_y, start_x, end_x = crop_box
    img_cropped = img_rgb[start_y:end_y, start_x:end_x]

    # 5. Inspect one pixel, in both channel orders
    bgr_value = img[pixel_y, pixel_x]
    rgb_value = img_rgb[pixel_y, pixel_x]

    # 6. Print the report
    print("Image Information")
    print("-----------------")
    print("Height:", height, "pixels")
    print("Width:", width, "pixels")
    print("Channels:", channels)
    print("Image Shape:", img.shape)
    print()
    print("Pixel Information")
    print("-----------------")
    print(f"Coordinate: (x={pixel_x}, y={pixel_y})")
    print("BGR Value:", bgr_value)
    print("RGB Value:", rgb_value)

    # 7. Show every version side by side
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    axes[0].imshow(img_rgb)
    axes[0].scatter([pixel_x], [pixel_y], s=200, facecolors="none",
                    edgecolors="red", linewidths=2)
    axes[0].set_title(f"Original {img_rgb.shape[1]} x {img_rgb.shape[0]}")

    axes[1].imshow(img_gray, cmap="gray")
    axes[1].set_title("Grayscale")

    axes[2].imshow(img_resized)
    axes[2].set_title(f"Resized {img_resized.shape[1]} x {img_resized.shape[0]}")

    axes[3].imshow(img_cropped)
    axes[3].set_title(f"Cropped {img_cropped.shape[1]} x {img_cropped.shape[0]}")

    for ax in axes:
        ax.axis("off")

    plt.show()

In [ ]:
explore_image(IMAGE_PATH)

### Make it yours

Every setting is an argument you can change. For example:

```python
explore_image(IMAGE_PATH, pixel_x=520, pixel_y=90, new_size=(128, 128), crop_box=(40, 160, 450, 600))
```

Try it in the empty cell below.

In [ ]:
# Your turn - change the numbers and run this cell
explore_image(IMAGE_PATH, pixel_x=520, pixel_y=90)

---

# Student Challenges

Work through these in the empty cells below. There is no grading - the point is to break things and see what happens.

### Challenge 1
Use a different image. Drop your own photo into this folder and change `IMAGE_PATH`.
Does everything still work? What is its shape?

### Challenge 2
Choose three different pixels and compare their values.
Pick one dark, one bright, and one strongly colored. Can you predict the numbers before you print them?

### Challenge 3
Resize the image to a different resolution. Try something very small, like `(50, 50)`.

### Challenge 4
Crop an interesting region from the image.

### Challenge 5
Compare the shape of:

* the original image
* the resized image
* the cropped image
* the grayscale image

Print all four shapes in one cell, one under the other.

> **What do these changes tell us about how the computer represents an image?**

In [ ]:
# Challenge 1 - your code here

In [ ]:
# Challenge 2 - your code here

In [ ]:
# Challenge 3 - your code here

In [ ]:
# Challenge 4 - your code here

In [ ]:
# Challenge 5 - your code here

---

## What Did We Learn?

* An image is represented as **numerical data**
* Images are stored as **NumPy arrays**
* Images have **height, width, and channels**
* **RGB** images usually have three channels
* **Grayscale** images usually have one channel
* Every **pixel** contains numerical information
* Pixel values can be inspected using **coordinates**, written as `image[y, x]`
* **Resizing** changes image dimensions
* **Cropping** selects a portion of an image
* Before using an image in AI, we first need to understand its numerical representation

And the trap worth remembering: OpenCV reads colors as **BGR**, Matplotlib expects **RGB**.

---

```text
An image
    -> is made of pixels
    -> pixels contain numbers
    -> those numbers are organized into arrays
    -> and this numerical representation is what computers use to process visual information.
```

---

> **Before a model can understand an image, we need to understand how that image is represented as data.**